# 02b - Conferencias da Silver (fora do caminho do job)

Celulas de exploracao e conferencia retiradas de `02_silver_promocao` em 16/09/2026 (plano C, passo 1.3). Rodar a mao, depois de uma carga. **A celula de amostra de idiomas exibe texto de postagem: a saida nao vai a print nem a documento.**

In [0]:
%python
dbutils.widgets.text("caso", "monark", "Caso")
dbutils.widgets.text("versao_pipeline", "v0.2.0-dev", "Versao do pipeline")

In [0]:
SELECT caso_slug, indicador, valor, detalhe
FROM silver.qc_resultado
WHERE versao_pipeline = :versao_pipeline AND NOT aprovado
ORDER BY caso_slug, indicador;

In [0]:
SELECT idioma, COUNT(*) AS n
FROM silver.v_deduplicado
WHERE caso_slug = 'arthur_do_val' AND idioma IS NOT NULL AND idioma <> 'pt'
GROUP BY idioma ORDER BY n DESC;

In [0]:
SELECT idioma, texto
FROM silver.v_deduplicado
WHERE caso_slug = 'arthur_do_val' AND idioma IN ('es', 'it', 'tl', 'ca')
ORDER BY rand()
LIMIT 15;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

SELECT postagem_id, COUNT(*) AS n
FROM silver.captura
GROUP BY postagem_id
HAVING COUNT(*) > 1
LIMIT 10;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

SELECT p.caso_slug,
       COUNT(DISTINCT p.postagem_id)  AS postagens,
       COUNT(c.captura_id)            AS capturas,
       COUNT(DISTINCT m.mencao_id)    AS mencoes,
       COUNT(DISTINCT h.hashtag)      AS hashtags_distintas,
       COUNT(DISTINCT k.classificacao_id) AS classificacoes
FROM silver.postagem p
LEFT JOIN silver.captura c          ON c.postagem_id = p.postagem_id
LEFT JOIN silver.mencao m           ON m.postagem_id = p.postagem_id
LEFT JOIN silver.postagem_hashtag h ON h.postagem_id = p.postagem_id
LEFT JOIN silver.classificacao k    ON k.postagem_id = p.postagem_id
GROUP BY p.caso_slug
ORDER BY p.caso_slug;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

WITH
pst AS (SELECT caso_slug, COUNT(*) AS postagens FROM silver.postagem GROUP BY caso_slug),
cap AS (SELECT p.caso_slug, COUNT(*) AS capturas
        FROM silver.captura c JOIN silver.postagem p ON p.postagem_id = c.postagem_id
        GROUP BY p.caso_slug),
men AS (SELECT p.caso_slug, COUNT(*) AS mencoes
        FROM silver.mencao m JOIN silver.postagem p ON p.postagem_id = m.postagem_id
        GROUP BY p.caso_slug),
hsh AS (SELECT p.caso_slug, COUNT(*) AS pares_postagem_hashtag
        FROM silver.postagem_hashtag h JOIN silver.postagem p ON p.postagem_id = h.postagem_id
        GROUP BY p.caso_slug)
SELECT pst.caso_slug, pst.postagens, cap.capturas, men.mencoes, hsh.pares_postagem_hashtag
FROM pst
LEFT JOIN cap ON cap.caso_slug = pst.caso_slug
LEFT JOIN men ON men.caso_slug = pst.caso_slug
LEFT JOIN hsh ON hsh.caso_slug = pst.caso_slug
ORDER BY pst.caso_slug;